# Exploração Inicial dos Dados

Nesta etapa, será realizada uma exploração inicial dos datasets para compreender sua estrutura, granularidade, períodos, variáveis, valores ausentes e possíveis inconsistências.

O objetivo é identificar pontos de atenção antes da etapa de preparação e tratamento dos dados.

In [ ]:
import pandas as pd

## 1. Carregamento dos dados

Os cinco arquivos disponíveis serão carregados para DataFrames distintos para análise individual.

In [ ]:
pesticides = pd.read_csv('../dados/pesticides.csv')
rainfall = pd.read_csv('../dados/rainfall.csv')
temp = pd.read_csv('../dados/temp.csv')
yield_df = pd.read_csv('../dados/yield_df.csv')
crop_yield = pd.read_csv('../dados/yield.csv')


## 2. Estrutura dos datasets

Inicialmente, serão analisadas as dimensões, colunas, tipos de dados e valores nulos de cada dataset.

In [ ]:
pesticides.info()

In [ ]:
rainfall.info()

In [ ]:
temp.info()

In [ ]:
yield_df.info()

In [ ]:
crop_yield.info()

## 3. Inspeção inicial dos registros

A visualização das primeiras linhas permite observar a estrutura dos registros e identificar possíveis particularidades nos dados.

In [ ]:
pesticides.head()

In [ ]:
rainfall.head()

In [ ]:
temp.head()

In [ ]:
yield_df.head()

In [ ]:
crop_yield.head()

In [ ]:
pesticides.shape

In [ ]:
pesticides.columns

In [ ]:
rainfall.shape

In [ ]:
rainfall.columns

In [ ]:
temp.shape

In [ ]:
temp.columns

In [ ]:
yield_df.shape

In [ ]:
yield_df.columns

In [ ]:
crop_yield.shape

In [ ]:
crop_yield.columns

## 4. Exploração — Pesticides

Investigação inicial da cobertura de áreas, período, itens disponíveis e estrutura dos dados de uso de pesticidas.

In [ ]:
print('Áreas:', pesticides['Area'].nunique())
print('Anos:', pesticides['Year'].nunique())
print('Itens:', pesticides['Item'].nunique())

print('\nItens:')
print(pesticides['Item'].unique())

print('\nAnos:')
print(pesticides['Year'].unique())

## 5. Exploração — Rainfall

Investigação da cobertura de áreas e anos, além da identificação de valores ausentes e possíveis marcadores de ausência de informação.

In [ ]:
print('Áreas:', rainfall[' Area'].nunique())
print('Anos:', rainfall['Year'].nunique())
print('Valores nulos:', rainfall['average_rain_fall_mm_per_year'].isna().sum())

In [ ]:
print('Anos:')
print(rainfall['Year'].unique())

In [ ]:
rainfall['average_rain_fall_mm_per_year'].unique()[:20]

## 6. Exploração — Temp

Investigação da cobertura de países e anos e verificação da existência de uma observação única de temperatura para cada combinação de país e ano.

In [ ]:
print('Países:', temp['country'].nunique())
print('Anos:', temp['year'].nunique())
print('Valores nulos:', temp['avg_temp'].isna().sum())

In [ ]:
print('Anos:')
print(temp['year'].unique())

In [ ]:
print('Países:', temp['country'].unique()[:20])

## 7. Exploração — Yield Dataset

Investigação da cobertura de áreas, culturas e anos do dataset enriquecido, incluindo a identificação de possíveis duplicidades na combinação `Area + Item + Year`.

In [ ]:
print('Áreas:', yield_df['Area'].nunique())
print('Culturas:', yield_df['Item'].nunique())
print('Anos:', yield_df['Year'].nunique())
print('Valores nulos:', yield_df.isna().sum())

In [ ]:
print('Áreas:')
print(yield_df['Area'].unique()[:20])

In [ ]:
print('Culturas:')
print(yield_df['Item'].unique())

In [ ]:
print('Anos:')
print(yield_df['Year'].unique())

## 8. Exploração — Crop Yield

Investigação da estrutura, cobertura temporal, culturas, elementos e unidades do dataset de produtividade agrícola.

In [ ]:
print('Áreas:', crop_yield['Area'].nunique())
print('Culturas:', crop_yield['Item'].nunique())
print('Anos:', crop_yield['Year'].nunique())
print('Elementos:', crop_yield['Element'].nunique())
print('Unidades:', crop_yield['Unit'].nunique())

In [ ]:
print('Elementos:')
print(crop_yield['Element'].unique())

print('\nUnidades:')
print(crop_yield['Unit'].unique())

In [ ]:
print('Anos:')
print(crop_yield['Year'].unique())

## 9. Investigação de duplicidades

Após a exploração inicial, foram identificadas possíveis inconsistências na granularidade dos dados.

A investigação a seguir busca compreender a origem dessas duplicidades antes de definir qualquer estratégia de tratamento.

### 9.1 Duplicidades em `yield_df`

Será verificado se a combinação `Area + Item + Year` representa uma chave única no dataset.

In [ ]:
yield_df.duplicated(
    subset=['Area', 'Item', 'Year']
).sum()

In [ ]:
yield_df['Year'].value_counts().sort_index()

In [ ]:
yield_df['Item'].value_counts()

In [ ]:
yield_df[
    yield_df.duplicated(
        subset=['Area', 'Item', 'Year'],
        keep=False
    )
].sort_values(['Area', 'Item', 'Year']).head(20)

In [ ]:
yield_df[
    yield_df.duplicated(
        subset=['Area', 'Item', 'Year'],
        keep=False
    )
].sort_values(['Area', 'Item', 'Year']).tail(20)

In [ ]:
yield_df[
    yield_df.duplicated(
        subset=['Area', 'Item', 'Year'],
        keep=False
    )
].drop(columns='avg_temp').duplicated().sum()

In [ ]:
yield_df.drop(columns='avg_temp').drop_duplicates().shape

Após identificar as duplicidades, foi realizado um teste removendo a coluna `avg_temp` para verificar se as diferentes temperaturas eram a única causa das repetições.

O resultado mostrou que as duplicidades permaneceram. Isso está relacionado ao fato de que a coluna `Unnamed: 0` possui valores distintos para cada registro e funciona como um identificador técnico do arquivo.

Portanto, `Unnamed: 0` não representa uma variável analítica e deverá ser avaliada para remoção durante a etapa de tratamento dos dados.

### 9.2 Duplicidades em `temp`

Será verificado se a combinação `country + year` representa uma chave única no dataset de temperatura.

In [ ]:
temp.duplicated(
    subset=['country', 'year']
).sum()

In [ ]:
temp[
    temp.duplicated(
        subset=['country', 'year'],
        keep=False
    )
].sort_values(['country', 'year']).head(20)

### 9.3 Investigação das observações de temperatura

Após identificar múltiplas observações para algumas combinações de país e ano, será analisado um caso específico para compreender a estrutura desses registros.

In [ ]:
temp[
    (temp['country'] == 'Argentina') &
    (temp['year'] == 1990)
]

In [ ]:
temp[
    (temp['country'] == 'Argentina') &
    (temp['year'] == 1990)
].shape

### 9.4 Distribuição da quantidade de observações

Será analisada a quantidade de registros existente para cada combinação de país e ano, buscando verificar se o número de observações é constante ou varia entre os grupos.

In [ ]:
temp.groupby(['country', 'year']).size().value_counts().sort_index()

In [ ]:
temp.groupby(['country', 'year']).size().sort_values(ascending=False).head(10)

In [ ]:
temp.groupby(['country', 'year'])['avg_temp'].agg(
    ['count', 'size']
).sort_values('count').head(20)

In [ ]:
temp.groupby(['country', 'year'])['avg_temp'].agg(
    ['count', 'size']
)['count'].value_counts().sort_index()

## 10. Considerações iniciais

A exploração identificou diferenças de granularidade, cobertura temporal, valores ausentes e duplicidades entre os datasets.

Entre os principais pontos de atenção está a existência de múltiplas observações de temperatura para determinadas combinações de país e ano, o que também está relacionado à duplicidade observada em `yield_df`.

Esses pontos serão tratados e validados na etapa de preparação dos dados, mantendo os arquivos originais preservados.